In [1]:
import sys
sys.path.append('..')
from get_cutouts import get_cutout

from astropy.table import Table
from astropy.visualization.wcsaxes import SphericalCircle
from astropy import units as u

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

from astropy.coordinates import SkyCoord, match_coordinates_sky
import matplotlib.patches as patches

import os

from tqdm import tqdm

In [2]:
loa_table = Table.read('/pscratch/sd/d/dbustos/rot_curves/shredded_VI.fits')

loa_table[:5]

TARGETID,SGA_ID,TARGET_RA,TARGET_DEC,Z,Z_ERR,ZWARN,DELTACHI2,DIST,DIST_R26,PA,C_TO_F_ANGLE,ANGLE_OFF_AXIS,Selection,ZERR_MOD,unique_obs,Velocity,V_err,Z_center,VI,deproj_dist,deproj_r26
int64,int64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,int64,float64,float64
6521555517440,1157225,204.22749,32.09493,0.010016011261740405,0.010016011261740405,0,71.96394567680545,0.002217102021260331,0.36627155666222244,65.12960815429688,114.35180431415512,49.22219615985824,1,0.010016039025980373,3.0,-2.9046128078129714,4248.5806387633265,0.010025824882245725,0,0.002881332554681444,0.4760043290496788
67815440646152,993764,131.54147036553024,19.348963933595506,0.6888593799981794,0.6888593799981794,4,7.544203393161297,0.004898285806031614,0.44172182227817786,9.1672945022583,188.95042393619124,179.78312943393294,1,0.688859381126886,3.0,199184.86718681874,206561.79926844206,0.014690024421240908,0,0.00677902926522777,0.6113251204383032
169678387281926,1430874,257.7495971,56.9324928,0.028921517883227336,0.028921517883227336,0,121857.89663615823,0.003250448915035935,0.31250071154567677,6.557606220245361,206.22810010342067,199.6704938831753,1,0.02892152786179073,3.0,158.1223507935872,12147.438163427121,0.028379120392506063,0,0.008096415535868652,0.7783957484224773
234526940856326,5001181,151.1772061023157,1.833743864948717,0.03108437229025544,0.03108437229025544,0,13100.219412088394,0.001016580293940661,0.34937980530094326,107.75557708740234,118.54239042958262,10.786813342180281,1,0.031084381613582526,4.0,20.42217921981953,20799.955384114714,0.06202829814864271,0,0.0031839814928551905,1.0942754258430336
234563162865665,632204,150.23137029728193,3.151162371010314,1.2707527411865653,1.2707527411865653,4,1.0103464424610138,0.011775468159487008,0.8630857807133053,159.4539031982422,322.60583622868717,163.15193303044498,1,1.270752742292688,23.0,361581.49656812777,381265.8408514983,0.05075534924045021,0,0.013838610970700105,1.014304330992709


In [3]:
SGA = Table.read('/global/cfs/cdirs/cosmo/data/sga/2020/SGA-2020.fits', 'ELLIPSE')

SGA_dict = {}
for i in range(len(SGA)):
    SGA_dict[SGA['SGA_ID'][i]] = i

In [4]:
manual_vi = np.load('/pscratch/sd/d/dbustos/VI/ids_to_remove/manual_VI_ids.npy')

In [5]:
loa_table['manual_VI'] = 0

In [6]:
for i in tqdm(manual_vi):
    loa_table['manual_VI'][loa_table['SGA_ID']==i] = 1

100%|██████████| 191/191 [00:00<00:00, 7269.16it/s]


In [7]:
target_galaxies = loa_table[(loa_table['manual_VI'] == 1) & (loa_table['deproj_r26'] < 1.1)]

In [8]:
# fig1 = plt.figure(figsize=(7,5))
colors = plt.get_cmap('Set2')
fiber_radius = (107./70) * u.arcsec

# for each rotation curve galaxy, grab cut out, draw fibers on image, and save image
for sga_id in tqdm(np.unique(target_galaxies['SGA_ID'])[:0]):
    
    targ_list = target_galaxies[target_galaxies['SGA_ID']==sga_id]

    num_targs = len(targ_list)
    
    ra, dec = float(SGA['RA'][SGA_dict[sga_id]]), float(SGA['DEC'][SGA_dict[sga_id]])
    
    # D26 in arcmin
    d26 = SGA['D26'][SGA_dict[sga_id]]

    pix = int(2 * d26*60/0.262)

    dr = 10

    if (pix < 2500):
        npix = np.minimum(pix,1024)

    elif (pix > 2500):
        npix = np.minimum(pix,3000)

    try:
        img_file, wcs = get_cutout(sga_id, ra, dec, dr=dr, dir='/pscratch/sd/d/dbustos/loa_cutouts/cutouts/',size=npix)
        img = mpl.image.imread(img_file)

    except Exception:
        print(sga_id, ':img download error 1')
        os.remove(img_file)

        try:
            img_file, wcs = get_cutout(sga_id, ra, dec, dr=dr, dir='/pscratch/sd/d/dbustos/loa_cutouts/cutouts/',size=npix)
            img = mpl.image.imread(img_file)

        except Exception:
            print(sga_id, ':img download error 1')
            os.remove(img_file)
            continue
    # try:
    #     img_file, wcs = get_cutout(sga_id, ra, dec, dr=dr, dir='/pscratch/sd/d/dbustos/loa_cutouts/cutouts/',size=npix)
    #     img = mpl.image.imread(img_file)

    # except Exception:
    #     print(sga_id, ':img download error 1')

    #     try:
    #         dr = 11
    #         img_file, wcs = get_cutout(sga_id, ra, dec, dr=dr, dir='/pscratch/sd/d/dbustos/loa_cutouts/cutouts/',size=npix)
    #         img = mpl.image.imread(img_file)
            
    #     except Exception :
    #         print(sga_id, ':img download error 2')
    #         continue
        
        #continue

    num_slices = int(num_targs/8)
    
    for slices in range(num_slices):
        fig1 = plt.figure(figsize=(7,5))
    
        ax = fig1.add_subplot(111, projection=wcs)
        ax.imshow(np.flip(img, axis=0))
        ax.set(xlabel='ra', ylabel='dec')
        ax.text(int(0.02*npix), int(0.85*npix), 'SGA_ID: {}'.format(sga_id), fontsize=9, color='yellow')
        overlay = ax.get_coords_overlay('icrs')
        overlay.grid(color='white', ls='dotted');
    
        # Add the location of the DESI fibers.
        # SDSS fibers are 2" diameter, DESI is 107 um with 70 um/" plate scale.
        r1 = SphericalCircle((ra * u.deg, dec * u.deg), fiber_radius,
                             edgecolor='black', facecolor='none', alpha=0.8, lw=3,
                             transform=ax.get_transform('icrs'))
        r2 = SphericalCircle((ra * u.deg, dec * u.deg), fiber_radius,
                             edgecolor='red', facecolor='none', alpha=0.8, lw=2,
                             transform=ax.get_transform('icrs'))
        ax.add_patch(r1)
        ax.add_patch(r2)
    
        
        start_idx = slices * 8
        end_idx = start_idx + 8 
        batch = targ_list[start_idx:end_idx]
        
        legend_handles = [None] * len(batch)
        location = {}
    
        for i, targ in enumerate(batch):
            idx = start_idx + i 
            targ_ra, targ_dec = targ['TARGET_RA'], targ['TARGET_DEC']
    
            if (targ_ra, targ_dec) not in location:
                location[(targ_ra, targ_dec)] = colors(len(location))
            
            #edgecolor2 = colors(i)
            edgecolor2 = location[(targ_ra, targ_dec)]
    
            # Add the location of the DESI fibers.
            # SDSS fibers are 2" diameter, DESI is 107 um with 70 um/" plate scale.
            r1 = SphericalCircle((targ_ra * u.deg, targ_dec * u.deg), fiber_radius,
                                 edgecolor='black', facecolor='none', alpha=1, lw=3,
                                 transform=ax.get_transform('icrs'))
            r2 = SphericalCircle((targ_ra * u.deg, targ_dec * u.deg), fiber_radius,
                                 edgecolor=edgecolor2, facecolor='none', alpha=1, lw=2,
                                 transform=ax.get_transform('icrs'))
            ax.add_patch(r1)
            ax.add_patch(r2)
    
            # ax.text(ra, dec, i, transform=ax.get_transform('icrs'), color='white', fontsize=6)
            
            legend_handles[i] = patches.Circle((0,0), facecolor = edgecolor2, ec = edgecolor2, lw = 2, label =  str(targ['TARGETID']))
        
        ax.legend(handles = legend_handles, title = 'SGA ID: {}'.format(sga_id), loc = 'upper right', bbox_to_anchor = (2, 1))
        fig1.subplots_adjust(top=0.85, right=0.85, bottom=0.15, left=0.15)
        plt.show()
    
    # fig1.savefig('/pscratch/sd/d/dbustos/loa_cutouts/cutouts_fibers/' + 'cutouts_sga_{}.png'.format(sga_id), bbox_inches = 'tight', dpi=120)
    
    # fig1.clear()

0it [00:00, ?it/s]


In [10]:
target_ids = np.loadtxt('target_ids_to_remove.txt')

In [16]:
target_galaxies['large_galaxy_VI'] = 0

In [17]:
for i in tqdm(target_ids):
    # mark as 1 to remove
    target_galaxies['large_galaxy_VI'][target_galaxies['TARGETID'] == i] = 1

100%|██████████| 774/774 [00:00<00:00, 27229.80it/s]


In [21]:
gals_to_remove = [75960, 178440, 572447, 868712, 936379, 972325, 2000022, 2000069]

In [23]:
for i in tqdm(gals_to_remove):
    # mark as 1 to remove
    target_galaxies['large_galaxy_VI'][target_galaxies['SGA_ID'] == i] = 1

100%|██████████| 8/8 [00:00<00:00, 10764.98it/s]


In [28]:
loa_table = target_galaxies[target_galaxies['large_galaxy_VI'] == 0]

In [31]:
# find new number unique obs
loa_table['unique_obs'] = np.nan

for i in tqdm(np.unique(loa_table['SGA_ID'])):
    
    fiber = loa_table['SGA_ID'] == i 

    sga_id = SGA_dict[i] 

    obs = loa_table[fiber]

    # identify all points on semimajor axis
    on_major = obs[obs['ANGLE_OFF_AXIS'] == 0]
    if len(on_major) == 0:
        unique_on = 0

    else:
        if (SGA['PA'][sga_id] < 45) or (SGA['PA'][sga_id] > 135):

            left_index = on_major['TARGET_DEC'] - SGA['DEC'][sga_id] > 0

        else:
            left_index = on_major['TARGET_RA'] - SGA['RA'][sga_id] > 0
                    
        left = on_major[left_index]
        right = on_major[~left_index]
        
        left['deproj_r26'] = np.round(left['deproj_r26'], 6)
        right['deproj_r26'] = np.round(right['deproj_r26'], 6)
    
        left_on_major = left.group_by('deproj_r26')
        right_on_major = right.group_by('deproj_r26')

        left_len = len(left_on_major.groups) if len(left) > 0 else 0 
        right_len = len(right_on_major.groups) if len(right) > 0 else 0 

        unique_on = left_len + right_len

    # identify all points off major axis
    off_major = obs[obs['ANGLE_OFF_AXIS'] != 0]
    
    # find number of unique points off axis
    num_off_major = len(off_major)

    if num_off_major == 0:
        unique_off = 0
    else:
        off_major = off_major.group_by(['deproj_r26','ANGLE_OFF_AXIS'])
        unique_off = len(off_major.groups)

    total_obs = unique_on + unique_off

    loa_table['unique_obs'][fiber] = total_obs

100%|██████████| 163/163 [00:01<00:00, 141.17it/s]


In [33]:
# redo criteria cut
center_dist_lim = 0.001
unique_dist_lim = 0.01

In [34]:
# redo criteria cut
#empty column to classify galaxy
loa_table['Selection'] = 0

#for each unique SGA ID
for i in tqdm(np.unique(loa_table['SGA_ID'])):

    #identify all the galaxies and targets
    obs_id = loa_table['SGA_ID'] == i
    
    #makes a table of the targets corresponding to this galaxy
    obs = loa_table[obs_id]

    sga_id = SGA_dict[i] 

    criteria_a = False
    criteria_b = False

#if the galaxy has more than three observations
    if len(obs) >= 3:

#-----
# a
#-----
        #check to see if there is a center observation
        if np.any(obs['deproj_r26'] <= center_dist_lim):
            # test_counter += 1
        
            # check how many unique points on semimajor axis
            not_center = obs[obs['deproj_r26'] > unique_dist_lim]

            if len(not_center) >= 2 and (np.max(not_center['deproj_r26']) - np.min(not_center['deproj_r26'])) >= unique_dist_lim:
                
                #since there 2 unique points, classify it as viable galaxy
                loa_table['Selection'][obs_id] = 1
                criteria_a = True

# #----
# # b
# #----
        elif len(obs) >= 10:
            distances = sorted(obs['deproj_r26'])
            counter = 0

            for i in range(len(distances)-1):
                if distances[i+1] - distances[i] > unique_dist_lim:
                    counter += 1
            
            if counter >= 10:
                loa_table['Selection'][obs_id] = 2
                criteria_b = True

        
#----
# c
#----
          
        elif not (criteria_a or criteria_b):
            
            # identify all points on semimajor axis
            on_major = obs[obs['ANGLE_OFF_AXIS'] == 0]

            # identify all points off major axis
            off_major = obs[obs['ANGLE_OFF_AXIS'] != 0]


            if len(on_major) > 0:
            
                #split targets on semimajor axis onto either side of center galaxy
                if (SGA['PA'][sga_id] < 45) or (SGA['PA'][sga_id] > 135):

                    left_index = on_major['TARGET_DEC'] - SGA['DEC'][sga_id] > 0

                else:
                    left_index = on_major['TARGET_RA'] - SGA['RA'][sga_id] > 0
                
                left = on_major[left_index]
                right = on_major[~left_index]

            if len(left) > 0 and len(right) > 0:

                for j in range(len(left)):
                    
                    # check that there are 2 symmetric observations
                    if np.any(np.abs(right['deproj_r26'] - left['deproj_r26'][j]) < unique_dist_lim):

                    #check if there is a third point
                        if (np.any(np.abs(right['deproj_r26'] - left['deproj_r26'][j]) > unique_dist_lim) 
                            or np.any(np.abs(left['deproj_r26'] - left['deproj_r26'][j]) > unique_dist_lim)
                            or np.any(np.abs(off_major['deproj_r26'] - left['deproj_r26'][j]) > unique_dist_lim)):

                            #viable galaxy
                            loa_table['Selection'][obs_id] = 3
                            
    # if tf_loa['Selection'][obs_id][0] == 1:
    #     print(obs['SGA_ID'][0])

print(len(np.unique(loa_table['SGA_ID'][loa_table['Selection'] != 0])))

100%|██████████| 163/163 [00:00<00:00, 914.33it/s]

153


In [35]:
loa_table = loa_table[loa_table['Selection'] != 0]

In [36]:
# run code to find rotational velocity, but first get proper errors